In [12]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import genextreme
from scipy.special import gamma as gamma_func
import scipy
import pandas as pd
import time


save_files=True

directory = "data"
directory_results=directory+"/Results"

filelist = [
    "gev-mu=70,alpha=10,gamma=0.3,N=e5.csv",
    "gev-mu=70,alpha=10,gamma=0.3,N=e5b.csv",
    "gev-mu=70,alpha=10,gamma=0.3,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=0.1,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=0.2,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=0.5,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=0.7,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=0.8,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=0.9,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=1,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=1.1,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=1.2,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=1.3,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=1.4,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=1.5,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=1.6,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=1.7,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=1.8,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=1.9,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=2,N=e6.csv",
    "gev-mu=70,alpha=10,gamma=-0.3,N=e6.csv",
    "gev-mu=70,alpha=100,gamma=-0.3,N=e6.csv"
    ]

In [13]:
for filename in filelist:
    # Load Data

    start_time = time.time()
    filepath = directory+ "/" + filename
    print(f"Filepath: {filepath}")
    df = pd.read_csv(filepath, header=None)


Filepath: data/gev-mu=70,alpha=10,gamma=0.3,N=e5.csv
Filepath: data/gev-mu=70,alpha=10,gamma=0.3,N=e5b.csv
Filepath: data/gev-mu=70,alpha=10,gamma=0.3,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=0.1,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=0.2,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=0.5,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=0.7,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=0.8,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=0.9,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=1,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=1.1,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=1.2,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=1.3,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=1.4,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=1.5,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=1.6,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=1.7,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=1.8,N=e6.csv
Filepath: data/gev-mu=70,alpha=10,gamma=1.9,N=e

In [9]:
filename="gev-mu=70,alpha=10,gamma=0.3,N=e6.csv"
filepath = directory+ "/" + filename
print(f"Filepath: {filepath}")
df = pd.read_csv(filepath, header=None)

Filepath: data/gev-mu=70,alpha=10,gamma=0.3,N=e6.csv


In [10]:

def weibull_plotting_positions(n):
    ranks = np.arange(1, n + 1)
    P = ranks / (n + 1)
    return P

data = df.values[:,0]
x = np.sort(data)
init = [np.median(x), np.std(x), 0.1]


x = np.sort(data)
N = len(x)
P = weibull_plotting_positions(N)


def objective_initial(params):
    mu, alpha, gamma = params
    if alpha <= 0:
        return 1e12
    xi = genextreme.ppf(P, -gamma, loc=mu, scale=alpha)                     #minus? gamma
    return np.sum((x - xi) ** 2)


res = minimize(objective_initial, init, method="Nelder-Mead")
params = res.x
params


array([69.96955912, 10.20386927,  0.29257541])

In [11]:
k=0
beta=0.001
mu, alpha, gamma = params

# theoretical 
xi_vals = genextreme.ppf(P, -gamma, loc=mu, scale=alpha)

print(f"xi_vals len: {len(xi_vals)}")
#print("rejecting points...")

while k < N - 5:

    denom = xi_vals[-1] - xi_vals[k]

    #if denom < 1e-12:
    #   k += 1
    #    continue

    ratio = (xi_vals[k+1] - xi_vals[k]) / denom

    if ratio < beta:
        k += 1
    else:
        break

if k > 0:
    x_fit = x[k:]
    P_fit = P[k:]
else:
    x_fit = x
    P_fit = P
print(f"rejected {k} points, remaining {len(x_fit)} points")
N_remaining=len(x_fit)

xi_vals len: 1000000
rejected 999898 points, remaining 102 points


In [ ]:
# Helper Functions

def weibull_plotting_positions(n):
    ranks = np.arange(1, n + 1)
    P = ranks / (n + 1)
    return P


def gev_pdf(x, mu, alpha, gamma):

    if abs(gamma) < 1e-10:
        y = (x - mu) / alpha
        return (1.0 / alpha) * np.exp(-y) * np.exp(-np.exp(-y))

    inner = 1.0 + gamma * (x - mu) / alpha
    inner = np.maximum(inner, 1e-12)

    return (1.0 / alpha) * inner ** (-(1.0 / gamma + 1.0)) * np.exp(-inner ** (-1.0 / gamma))


def gev_cdf(x, mu, alpha, gamma):

    if abs(gamma) < 1e-10:
        y = (x - mu) / alpha
        return np.exp(-np.exp(-y))

    inner = 1.0 + gamma * (x - mu) / alpha
    inner = np.maximum(inner, 1e-12)

    return np.exp(-inner ** (-1.0 / gamma))



def order_statistic_variance(m, N, mu, alpha, gamma):

    x_min = genextreme.ppf(1e-6, -gamma, loc=mu, scale=alpha)
    x_max = genextreme.ppf(1 - 1e-6, -gamma, loc=mu, scale=alpha)

    x_grid = np.linspace(x_min, x_max, 800)

    #print("calc F and f...")
    F_vals = gev_cdf(x_grid, mu, alpha, gamma)
    f_vals = gev_pdf(x_grid, mu, alpha, gamma)

    coeff = scipy.special.comb(N=N, k=m, exact=True)*m

    #print("calc fm_vals...")
    fm_vals = coeff * (F_vals ** (m - 1)) * ((1 - F_vals) ** (N - m)) * f_vals

    #print("calc mean...")
    mean_m = np.trapzoid(x_grid * fm_vals, x_grid)                              #integrating not from -inf to inf?
    
    #print("calc variance...")
    var_m  = np.trapzoid((x_grid - mean_m) ** 2 * fm_vals, x_grid)

    return max(var_m, 1e-12)



# VWLS fit
def fit_vwls(data, max_iterations=20, beta=0.001):

    x = np.sort(data)
    N = len(x)
    P = weibull_plotting_positions(N)


    def objective_initial(params):
        mu, alpha, gamma = params
        if alpha <= 0:
            return 1e12
        xi = genextreme.ppf(P, -gamma, loc=mu, scale=alpha)                     #minus? gamma
        return np.sum((x - xi) ** 2)

    init = [np.median(x), np.std(x), 0.1]

    res = minimize(objective_initial, init, method="Nelder-Mead")
    params = res.x

    k = 0

    for iteration in range(max_iterations):
        print(f"Iteration {iteration+1}/{max_iterations}, params: mu={params[0]:.6f}, alpha={params[1]:.6f}, gamma={params[2]:.6f}")
        mu, alpha, gamma = params

        # theoretical 
        xi_vals = genextreme.ppf(P, -gamma, loc=mu, scale=alpha)

        #print(f"xi_vals len: {len(xi_vals)}")
        #print("rejecting points...")
        raise NotImplementedError("Stopping here to check xi_vals and rejection logic. Something weird is going on here at the rejection step.")
        while k < N - 5:

            denom = xi_vals[-1] - xi_vals[k]

            if denom < 1e-12:
                k += 1
                continue

            ratio = (xi_vals[k+1] - xi_vals[k]) / denom

            if ratio < beta:
                k += 1
            else:
                break

        if k > 0:
            x_fit = x[k:]
            P_fit = P[k:]
        else:
            x_fit = x
            P_fit = P
        print(f"rejected {k} points, remaining {len(x_fit)} points")
        N_remaining=len(x_fit)


        print("calculating weights...")
        variances = np.zeros(N_remaining, dtype=np.float64)

        for m in range(1, N_remaining + 1):
            try:
                variances[m-1] = order_statistic_variance(m, N_remaining, mu, alpha, gamma)
            except:
                variances[m-1] = 1.0

        w_fit = 1 / variances
        w_fit = w_fit / w_fit.sum()


        print("optimizing parameters...")
        def objective(params_opt):

            mu_opt, alpha_opt, gamma_opt = params_opt

            if alpha_opt <= 0:
                return 1e12

            xi = genextreme.ppf(P_fit, -gamma_opt, loc=mu_opt, scale=alpha_opt)

            residuals = x_fit - xi

            return np.sum(w_fit * residuals**2)

        result = minimize(
            objective,
            params,
            method='Nelder-Mead',
            options={'xatol':1e-6,'fatol':1e-8,'maxiter':2000}
        )

        new_params = result.x

        if np.max(np.abs(new_params - params)) < 1e-5:
            params = new_params
            break

        params = new_params

    mu, alpha, gamma = params

    return mu, max(alpha,1e-6), gamma,len(x_fit)



# MLE
def fit_mle(data):

    x = np.sort(data)
    c, mu, alpha = genextreme.fit(x)
    gamma = -c
    return mu, alpha, gamma,len(x)



# PWM
def fit_pwm(data):

    x = np.sort(data)
    n = len(x)
    P = weibull_plotting_positions(n)

    b0 = np.mean(x)
    b1 = np.mean(x * P)
    b2 = np.mean(x * P**2)

    c = (2*b1 - b0)/(3*b2 - b0) - np.log(2)/np.log(3)

    k = 7.859*c + 2.9554*c**2
    gamma = -k

    if abs(gamma) < 1e-6:

        alpha = (b0 - 2*b1)/(-np.log(2))
        mu = b0 + alpha*0.5772

    else:

        g = gamma_func(1 + k)

        denom = g*(1 - 2**(-k))

        if abs(denom) < 1e-10:
            return np.mean(x), np.std(x), 0

        alpha = k*(2*b1 - b0)/denom
        mu = b0 - alpha*(1 - g)/k

    return mu, max(alpha,1e-6), gamma,len(x)


# L-moments
def fit_lmoments(data):

    x = np.sort(data)
    n = len(x)

    c = np.arange(n)

    b0 = np.mean(x)
    b1 = np.sum(c/(n-1)*x)/n
    b2 = np.sum(c*(c-1)/((n-1)*(n-2))*x)/n

    l1 = b0
    l2 = 2*b1 - b0
    l3 = 6*b2 - 6*b1 + b0

    if abs(l2) < 1e-10:
        return np.mean(x), np.std(x), 0

    t3 = l3/l2

    z = 2/(3+t3) - np.log(2)/np.log(3)

    k = 7.859*z + 2.9554*z**2
    gamma = -k

    if abs(gamma) < 1e-6:

        alpha = l2/np.log(2)
        mu = l1 - 0.5772*alpha

    else:

        g = gamma_func(1+k)

        alpha = k*l2/(g*(1-2**(-k)))
        mu = l1 - alpha*(1-g)/k

    return mu, max(alpha,1e-6), gamma,len(x)

In [15]:
for filename in filelist:
    # Load Data

    start_time = time.time()
    filepath = directory+ "/" + filename
    print(f"Filepath: {filepath}")
    df = pd.read_csv(filepath, header=None)

    data = df.values[:,0]


    # Fit all methods
    fits = {}
    data_subset = data#[:1_000]

    fits['VWLS'] = fit_vwls(data_subset)
    print("VWLS fit completed.")

    fits['MLE']  = fit_mle(data_subset)
    #print("MLE fit completed.")

    fits['PWM']  = fit_pwm(data_subset)
    #print("PWM fit completed.")

    fits['L-moments'] = fit_lmoments(data_subset)
    #print("L-moments fit completed.")


    #save in file
    if save_files:
        with open(f"{directory_results}/calc_param_for_files.csv", "a") as f:
            f.write(f"#{filename}\n")
            for method,(mu,alpha,shape,n) in fits.items():
                f.write(f"{method}, mu={mu:.6f}, alpha={alpha:.6f}, gamma={shape:.6f}, n={n}\n")
        print(f"Parameters saved to {directory_results}/calc_param_for_files.csv")
        print(f"took time: {time.time() - start_time:.2f} seconds\n\n")

    

Filepath: data/gev-mu=70,alpha=10,gamma=0.3,N=e5.csv
Iteration 1/20, params: mu=69.996543, alpha=9.908583, gamma=0.299697
rejected 99899 points, remaining 101 points
calculating weights...
optimizing parameters...
Iteration 2/20, params: mu=137.988649, alpha=5.331150, gamma=0.367185
rejected 99912 points, remaining 88 points
calculating weights...
optimizing parameters...
Iteration 3/20, params: mu=168.556495, alpha=3.977185, gamma=0.397201
rejected 99917 points, remaining 83 points
calculating weights...
optimizing parameters...
Iteration 4/20, params: mu=181.549402, alpha=3.478613, gamma=0.410810
rejected 99919 points, remaining 81 points
calculating weights...
optimizing parameters...
Iteration 5/20, params: mu=186.423801, alpha=3.303078, gamma=0.416053
rejected 99920 points, remaining 80 points
calculating weights...
optimizing parameters...
Iteration 6/20, params: mu=188.823542, alpha=3.218939, gamma=0.418661
rejected 99920 points, remaining 80 points
calculating weights...
optimi

KeyboardInterrupt: 